# NLP Classification of Technical Newsgroup Posts - Modeling
notebook 2

Table of Content

- [Imports and Setup](#imports-and-setup)
- [Split strategy and leakage control](#split-strategy-and-leakage-control)
- 



## <a id="imports-and-setup"></a>Imports and Setup


This notebook continues from the EDA and preprocessing notebook.

It loads the processed 20 Newsgroups technical subset, including all text regimes, artifact diagnostics, and evaluation slices created during EDA.

The main goals of this notebook are:

1. Define a leakage-aware train/validation/test split strategy.
2. Train classical NLP baselines across text regimes.
3. Compare performance using macro F1, per-class metrics, confusion matrices, and slice analysis.
4. Inspect model features to check whether predictions rely on topic vocabulary or shortcut artifacts.
5. Prepare a comparable transformer evaluation.

# PLAN


## Split audit
## Choose/save split
## Baseline MVP: TF-IDF + LR on text_subject_body_clean
## Text regime comparison
## TF-IDF + NB/LR ablation
## Feature inspection
## Slice evaluation
## Error analysis
## Transformer
## Transformer error/explainability
## Final recommendation

In [39]:
# standard library 
import itertools
import json
import os
import re
import time
from datetime import datetime
from pathlib import Path

# core data 
import numpy as np
import pandas as pd

# visualisation
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.figure_factory as ff
import plotly.graph_objects as go
from plotly.subplots import make_subplots

import sklearn
from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedShuffleSplit, GroupShuffleSplit
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

from sklearn.metrics.pairwise import cosine_similarity
import random
from scipy.sparse import vstack

import platform

In [40]:
# global configs

RANDOM_STATE = 42

np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 200)

In [41]:
PROJECT_ROOT = Path(".")

DATA_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
OUTPUT_TABLES_DIR = PROJECT_ROOT / "outputs" / "tables"
OUTPUT_CONFIG_DIR = PROJECT_ROOT / "outputs" / "config"
OUTPUT_FIGURES_DIR = PROJECT_ROOT / "outputs" / "figures"

MODELING_OUTPUT_DIR = PROJECT_ROOT / "outputs" / "modeling"
MODELING_TABLES_DIR = MODELING_OUTPUT_DIR / "tables"
MODELING_CONFIG_DIR = MODELING_OUTPUT_DIR / "config"
MODELING_FIGURES_DIR = MODELING_OUTPUT_DIR / "figures"
MODELING_PREDICTIONS_DIR = MODELING_OUTPUT_DIR / "predictions"

for directory in [
    MODELING_OUTPUT_DIR,
    MODELING_TABLES_DIR,
    MODELING_CONFIG_DIR,
    MODELING_FIGURES_DIR,
    MODELING_PREDICTIONS_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

In [42]:
# load EAD manifest and config

manifest_path = OUTPUT_CONFIG_DIR / "eda_output_manifest.json"
config_path = OUTPUT_CONFIG_DIR / "eda_preprocessing_config.json"

with open(manifest_path, "r", encoding="utf-8") as f:
    eda_manifest = json.load(f)

with open(config_path, "r", encoding="utf-8") as f:
    preprocessing_config = json.load(f)

print("Loaded EDA manifest:")
print(json.dumps(eda_manifest, indent=2, ensure_ascii=False)[:1500])

print("\nLoaded preprocessing config keys:")
print(preprocessing_config.keys())

Loaded EDA manifest:
{
  "created_at": "2026-05-20T15:12:45",
  "processed_dataset": "data/processed/20ng_tech_preprocessed.parquet",
  "modeling_input_dataset": "data/processed/20ng_tech_modeling_input.parquet",
  "config": "outputs/config/eda_preprocessing_config.json",
  "environment": "outputs/config/eda_environment_info.json",
  "saved_tables": {
    "length_impact": "outputs/tables/preprocessing_length_impact.csv",
    "retention_summary": "outputs/tables/preprocessing_word_retention_summary.csv",
    "class_length_impact": "outputs/tables/preprocessing_class_length_impact.csv",
    "artifact_overall": "outputs/tables/artifact_overall.csv",
    "artifact_by_class": "outputs/tables/artifact_density_by_class.csv",
    "artifact_intensity_selected": "outputs/tables/artifact_intensity_selected.csv",
    "artifact_residuals": "outputs/tables/artifact_residuals_after_preprocessing.csv",
    "length_slice_by_regime": "outputs/tables/length_slice_by_regime.csv",
    "length_slice_pivot":

In [43]:
# load processed modeling dataset

modeling_input_path = Path(eda_manifest["modeling_input_dataset"])

if modeling_input_path.suffix == ".parquet":
    df = pd.read_parquet(modeling_input_path)
elif modeling_input_path.suffix == ".pkl":
    df = pd.read_pickle(modeling_input_path)
else:
    raise ValueError(f"Unsupported modelling input format: {modeling_input_path}")

df.shape

(6866, 17)

In [44]:
df.head()

,doc_id,original_index,label,label_id,text,text_raw,text_subject_body_clean,text_body_only_clean,text_subject_body_no_quotes,quote_density_slice,artifact_density_slice,text_raw_length_slice,text_subject_body_clean_length_slice,text_body_only_clean_length_slice,text_subject_body_no_quotes_length_slice,body_quote_line_ratio,structural_line_ratio
0,doc_00000,0,comp.sys.ibm.pc.hardware,2,From: k4bnc@cbnewsh.cb.att.com (john.a.siegel)\nSubject: Can't set COM4\nOrganization: AT&T\nDistribution: usa\nKeywords: G2K\nLines: 15\n\nI have been unable to get COM 4 to work - diagnostic pro...,From: k4bnc@cbnewsh.cb.att.com (john.a.siegel)\nSubject: Can't set COM4\nOrganization: AT&T\nDistribution: usa\nKeywords: G2K\nLines: 15\n\nI have been unable to get COM 4 to work - diagnostic pro...,Can't set COM4\n\nI have been unable to get COM 4 to work - diagnostic programs such as msd show\nnothing installed. I think the software options are OK - is there a known\nhardware conflict and/o...,I have been unable to get COM 4 to work - diagnostic programs such as msd show\nnothing installed. I think the software options are OK - is there a known\nhardware conflict and/or workaround for t...,Can't set COM4\n\nI have been unable to get COM 4 to work - diagnostic programs such as msd show\nnothing installed. I think the software options are OK - is there a known\nhardware conflict and/o...,no_detected_quotes,low_structural_artifacts,medium_80_255,medium_80_255,medium_80_255,medium_80_255,0.000000,0.000000
1,doc_00001,1,comp.sys.ibm.pc.hardware,2,"From: mars@carroll1.cc.edu (Sean Tyler Mars)\nSubject: Help: Blowing the stack\nExpires: 29 Apr 93 23:00:00 GMT\nOrganization: Carroll College-Waukesha, WI\nLines: 25\n\n\n\tHi everyone,\n\tI have...","From: mars@carroll1.cc.edu (Sean Tyler Mars)\nSubject: Help: Blowing the stack\nExpires: 29 Apr 93 23:00:00 GMT\nOrganization: Carroll College-Waukesha, WI\nLines: 25\n\n\n\tHi everyone,\n\tI have...","Help: Blowing the stack\n\nHi everyone,\n I have a question regarding my stack on my pc. I am programming\nin Turbo C 3.0 and my program is rather large (model large too). I keep\ngetting errors t...","Hi everyone,\n I have a question regarding my stack on my pc. I am programming\nin Turbo C 3.0 and my program is rather large (model large too). I keep\ngetting errors that I am running out of mem...","Help: Blowing the stack\n\nHi everyone,\n I have a question regarding my stack on my pc. I am programming\nin Turbo C 3.0 and my program is rather large (model large too). I keep\ngetting errors t...",no_detected_quotes,low_structural_artifacts,medium_80_255,medium_80_255,medium_80_255,medium_80_255,0.000000,0.000000
2,doc_00002,2,comp.sys.mac.hardware,3,"From: peter@ferranti.com (peter da silva)\nSubject: Re: DCC and MiniDisc: next DAT/DDS like story?\nOrganization: Xenix Support, FICC\nLines: 15\n\nIn article <C50CMD.1zz@newcastle.ac.uk> Tor-Olav...","From: peter@ferranti.com (peter da silva)\nSubject: Re: DCC and MiniDisc: next DAT/DDS like story?\nOrganization: Xenix Support, FICC\nLines: 15\n\nIn article <C50CMD.1zz@newcastle.ac.uk> Tor-Olav...","Re: DCC and MiniDisc: next DAT/DDS like story?\n\nIn article < <EMAIL> > <EMAIL> (Tor-Olav Berntzen) writes:\n> Another thing, why a SCSI interface ?\n\nBecause SCSI works well with removable medi...","In article < <EMAIL> > <EMAIL> (Tor-Olav Berntzen) writes:\n> Another thing, why a SCSI interface ?\n\nBecause SCSI works well with removable media, and works well with large\ncapacity devices. Th...","Re: DCC and MiniDisc: next DAT/DDS like story?\n\nIn article < <EMAIL> > <EMAIL> (Tor-Olav Berntzen) writes:\n\nBecause SCSI works well with removable media, and works well with large\ncapacity de...",low_quote_<25%,medium_structural_artifacts,medium_80_255,medium_80_255,medium_80_255,medium_80_255,0.066667,0.095238
3,doc_00003,3,sci.electronics,6,"From: Wayne Alan Martin <wm1h+@andrew.cmu.edu>\nSubject: Re: Dayton Hamfest\nOrganization: Senior, Electrical and Computer Engin

In [45]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6866 entries, 0 to 6865
Data columns (total 17 columns):
 #   Column                                    Non-Null Count  Dtype   
---  ------                                    --------------  -----   
 0   doc_id                                    6866 non-null   object  
 1   original_index                            6866 non-null   int64   
 2   label                                     6866 non-null   object  
 3   label_id                                  6866 non-null   int64   
 4   text                                      6866 non-null   object  
 5   text_raw                                  6866 non-null   object  
 6   text_subject_body_clean                   6866 non-null   object  
 7   text_body_only_clean                      6866 non-null   object  
 8   text_subject_body_no_quotes               6866 non-null   object  
 9   quote_density_slice                       6866 non-null   object  
 10  artifact_density_slice  

In [46]:
# validate required columns

required_columns = [
    "doc_id",
    "original_index",
    "label",
    "label_id",
    "text",
    "text_raw",
    "text_subject_body_clean",
    "text_body_only_clean",
    "text_subject_body_no_quotes",
    "quote_density_slice",
    "artifact_density_slice",
    "body_quote_line_ratio",
    "structural_line_ratio",
]

missing_columns = [col for col in required_columns if col not in df.columns]

if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

print("All required columns are present.")

All required columns are present.


In [47]:
# define text regimes and labels
TEXT_REGIMES = [
    "text_raw",
    "text_subject_body_clean",
    "text_body_only_clean",
    "text_subject_body_no_quotes",
]

MAIN_TEXT_REGIME = "text_subject_body_clean"
TARGET_COL = "label"
TARGET_ID_COL = "label_id"
ID_COL = "doc_id"

LABELS = sorted(df[TARGET_COL].unique())

LABELS

['comp.graphics',
 'comp.os.ms-windows.misc',
 'comp.sys.ibm.pc.hardware',
 'comp.sys.mac.hardware',
 'comp.windows.x',
 'sci.crypt',
 'sci.electronics']

In [48]:
# sanity check
text_regime_summary = []

for col in TEXT_REGIMES:
    text_regime_summary.append({
        "text_regime": col,
        "missing_texts": df[col].isna().sum(),
        "empty_texts": df[col].fillna("").str.strip().eq("").sum(),
        "mean_words": df[col].fillna("").str.split().str.len().mean(),
        "median_words": df[col].fillna("").str.split().str.len().median(),
    })

text_regime_summary = pd.DataFrame(text_regime_summary)

display(
    text_regime_summary.style.format({
        "mean_words": "{:.1f}",
        "median_words": "{:.1f}",
    })
)

,text_regime,missing_texts,empty_texts,mean_words,median_words
0,text_raw,0,0,242.5,154.0
1,text_subject_body_clean,0,0,217.1,128.0
2,text_body_only_clean,0,28,211.4,122.0
3,text_subject_body_no_quotes,0,0,180.3,94.0


In [49]:
# optional eda tables for refernce 

def load_table_if_exists(path):
    path = Path(path)
    if path.exists():
        return pd.read_csv(path)
    print(f"File not found: {path}")
    return None

eda_tables = {}

for table_name, table_path in eda_manifest.get("saved_tables", {}).items():
    path = Path(table_path)
    if path.exists():
        eda_tables[table_name] = pd.read_csv(path)
    else:
        print(f"Missing table: {table_name} -> {path}")

print(f"Loaded {len(eda_tables)} EDA tables.")
print(sorted(eda_tables.keys()))

Loaded 20 EDA tables.
['artifact_by_class', 'artifact_intensity_selected', 'artifact_overall', 'artifact_residuals', 'artifact_slice_summary', 'centroid_similarity_df', 'class_length_impact', 'length_impact', 'length_slice_by_regime', 'length_slice_pivot', 'pair_similarity_df', 'quote_by_class', 'quote_slice_summary', 'retention_summary', 'shared_terms_pairs', 'top_terms_overall_text_body_only_clean', 'top_terms_overall_text_raw', 'top_terms_overall_text_subject_body_clean', 'top_terms_overall_text_subject_body_no_quotes', 'vocab_diagnostics']


In [50]:
# save modeling notebook config

modeling_config = {
    "created_at": datetime.now().isoformat(timespec="seconds"),
    "notebook": "02_modeling_evaluation.ipynb",
    "random_state": RANDOM_STATE,
    "dataset_path": str(modeling_input_path),
    "target_col": TARGET_COL,
    "target_id_col": TARGET_ID_COL,
    "id_col": ID_COL,
    "text_regimes": TEXT_REGIMES,
    "main_text_regime": MAIN_TEXT_REGIME,
    "labels": LABELS,
    "planned_split_strategy": [
        "Attempt thread/quote-aware grouping using Message-ID, References, In-Reply-To, and quote fingerprints.",
        "If feasible, use group-aware train/validation/test split.",
        "If not feasible, use stratified split and report quoted-thread overlap as a validity threat.",
    ],
    "primary_metric": "macro_f1",
    "secondary_metrics": ["accuracy", "per_class_precision", "per_class_recall", "per_class_f1"],
}

modeling_config_path = MODELING_CONFIG_DIR / "modeling_config_initial.json"

with open(modeling_config_path, "w", encoding="utf-8") as f:
    json.dump(modeling_config, f, indent=2, ensure_ascii=False)

print(f"Saved modelling config to: {modeling_config_path}")

Saved modelling config to: outputs/modeling/config/modeling_config_initial.json


In [51]:
# env shapshot
environment_info = {
    "created_at": datetime.now().isoformat(timespec="seconds"),
    "python_version": platform.python_version(),
    "platform": platform.platform(),
    "pandas_version": pd.__version__,
    "numpy_version": np.__version__,
    "sklearn_version": sklearn.__version__,
}

environment_path = MODELING_CONFIG_DIR / "modeling_environment_info.json"

with open(environment_path, "w", encoding="utf-8") as f:
    json.dump(environment_info, f, indent=2, ensure_ascii=False)

environment_info

{'created_at': '2026-05-20T15:15:25',
 'python_version': '3.13.5',
 'platform': 'macOS-26.3.1-arm64-arm-64bit-Mach-O',
 'pandas_version': '2.3.1',
 'numpy_version': '2.3.1',
 'sklearn_version': '1.7.1'}

## <a id="split-strategy-and-leakage-control"></a>Split strategy and leakage control


A standard random stratified split is not automatically sufficient for this dataset.

The main risk is not class imbalance, but possible content overlap through thread structure and quoted replies. During EDA, many posts were found to contain quoted previous messages. This means that two different documents may share parts of the same discussion thread. If related replies are split across train, validation, and test sets, a model may partially benefit from repeated quoted fragments rather than generalizing to genuinely unseen discussions.

This is a softer form of leakage than direct label leakage. The model is not necessarily seeing the label itself, but it may see repeated context from the same conversation across different splits. This could inflate validation or test performance, especially for lexical models such as TF-IDF.

To reduce this risk, I first attempt a graph-based grouping strategy. Each document is treated as a node. Edges between documents are created using two signals:

1. **Thread metadata**: `Message-ID`, `References`, and `In-Reply-To` headers, when available.
2. **Quote fingerprints**: normalized hashes of meaningful `>`-style quoted lines shared between documents.

Documents connected by either thread metadata or shared quote fingerprints are assigned to the same connected component. These components are then used as groups for a group-aware train/validation/test split, so that related posts stay in the same partition.

This approach is exploratory and will be audited before use. If the resulting groups are too sparse, too large, or too class-imbalanced for a reliable split, the fallback will be a standard stratified split with the quote/thread overlap risk reported explicitly as a validity limitation. In either case, the modelling stage will also compare preprocessing regimes, including a quote-reduced representation, to test how much performance depends on quoted context.

### Parse thread metadata  

The first grouping signal comes from top-level Usenet thread headers.

For this step, I parse only the initial header block of each raw document. Quoted or embedded headers inside the message body are intentionally ignored, because they may belong to previous messages rather than to the current document.

The goal is to extract:

- `Message-ID`: the current document identifier;
- `References`: previous messages in the thread;
- `In-Reply-To`: direct reply target, when available.

This metadata is not used as a model feature. It is used only to construct possible leakage-control groups for splitting.

In [52]:
# header parcer

def extract_top_header_block(text):
    """
    Extract the initial top-level header block from a raw Usenet post.

    The header block is assumed to end at the first blank line.
    """
    if pd.isna(text):
        return []

    lines = str(text).splitlines()
    header_lines = []

    for line in lines:
        if line.strip() == "":
            break
        header_lines.append(line)

    return header_lines


def parse_top_headers(text):
    """
    Parse top-level RFC-style headers from the initial header block.

    Handles folded continuation lines that start with whitespace.
    Returns a dictionary with lowercase header names.
    """
    header_lines = extract_top_header_block(text)

    headers = {}
    current_key = None

    for line in header_lines:
        header_match = re.match(r"^([A-Za-z][A-Za-z0-9\-]*)\s*:\s*(.*)$", line)

        if header_match:
            key = header_match.group(1).lower()
            value = header_match.group(2).strip()

            if key not in headers:
                headers[key] = value
            else:
                headers[key] = headers[key] + " " + value

            current_key = key

        elif current_key is not None and line.startswith((" ", "\t")):
            # folded header continuation
            headers[current_key] = headers[current_key] + " " + line.strip()

    return headers

In [53]:
# message id extraction helpers

MESSAGE_ID_PATTERN = re.compile(r"<[^<>]+>")


def normalize_message_id(message_id):
    """
    Normalize a Message-ID-like string.
    """
    if message_id is None or pd.isna(message_id):
        return None

    message_id = str(message_id).strip().lower()
    message_id = re.sub(r"\s+", "", message_id)

    return message_id if message_id else None


def extract_message_ids(value):
    """
    Extract all <...> message identifiers from a header value.
    """
    if value is None or pd.isna(value):
        return []

    ids = MESSAGE_ID_PATTERN.findall(str(value))
    ids = [normalize_message_id(mid) for mid in ids]
    ids = [mid for mid in ids if mid is not None]

    # preserve order while removing duplicates
    return list(dict.fromkeys(ids))


def first_or_none(items):
    return items[0] if len(items) > 0 else None

In [54]:
# parce thread fields into df columns

parsed_headers = df["text_raw"].apply(parse_top_headers)

df["message_id"] = parsed_headers.apply(
    lambda h: first_or_none(extract_message_ids(h.get("message-id")))
)

df["references_ids"] = parsed_headers.apply(
    lambda h: extract_message_ids(h.get("references"))
)

df["in_reply_to_ids"] = parsed_headers.apply(
    lambda h: extract_message_ids(h.get("in-reply-to"))
)

df["thread_reference_ids"] = df.apply(
    lambda row: list(dict.fromkeys(row["references_ids"] + row["in_reply_to_ids"])),
    axis=1,
)

df["has_message_id"] = df["message_id"].notna()
df["has_references"] = df["references_ids"].apply(lambda x: len(x) > 0)
df["has_in_reply_to"] = df["in_reply_to_ids"].apply(lambda x: len(x) > 0)
df["has_any_thread_reference"] = df["thread_reference_ids"].apply(lambda x: len(x) > 0)

,field,documents_with_field,share
0,Message-ID,0,0.0%
1,References,0,0.0%
2,In-Reply-To,47,0.7%
3,References or In-Reply-To,47,0.7%


In [57]:
# Посмотрим все уникальные header-поля в датасете
all_header_keys = set()
for text in df["text"].iloc[:100]:
    h = parse_top_headers(text)
    all_header_keys.update(h.keys())

print(sorted(all_header_keys))

['distribution', 'expires', 'from', 'in-reply-to', 'keywords', 'lines', 'news-software', 'nntp-posting-host', 'organization', 'originator', 'reply-to', 'subject', 'summary', 'x-newsreader']


In [58]:
print(df["text"].iloc[0][:300])

From: k4bnc@cbnewsh.cb.att.com (john.a.siegel)
Subject: Can't set COM4
Organization: AT&T
Distribution: usa
Keywords: G2K
Lines: 15

I have been unable to get COM 4 to work - diagnostic programs such as msd show
nothing installed.  I think the software options are OK - is there a known
hardware conf


In [36]:
import os
import glob

# Путь где sklearn кэширует данные
base_path = os.path.expanduser("~/scikit_learn_data")

# Папки train и test
train_path = os.path.join(base_path, "20news-bydate-train")
test_path  = os.path.join(base_path, "20news-bydate-test")

# Проверим что есть
print(os.listdir(base_path))

['cal_housing_py3.pkz', 'lfw_home', '20news-bydate_py3.pkz']


In [61]:
import urllib.request
import tarfile
import os

# Original archive with full headers
url = "http://qwone.com/~jason/20Newsgroups/20news-bydate.tar.gz"
archive_path = os.path.expanduser("~/scikit_learn_data/20news-bydate.tar.gz")
extract_path = os.path.expanduser("~/scikit_learn_data/20news-bydate-raw/")

# Download (~14 MB)
if not os.path.exists(archive_path):
    print("Downloading...")
    urllib.request.urlretrieve(url, archive_path)
    print("Done.")

# Unzip
if not os.path.exists(extract_path): 
    print("Extracting...") 
    with tarfile.open(archive_path, "r:gz") as tar: 
        tar.extractall(extract_path) 
    print("Done.")

# Check structure
print(os.listdir(extract_path))

['20news-bydate-test', '20news-bydate-train']
